# Reviewer Rebuttal Experiments (Non-Causal)

This notebook implements requested reviewer analyses without causal modeling.

Covered items:
1) top_p ablation (temperature x top_p main table)
2) significance tests / confidence intervals for Exp 2-B and Exp 2-C style comparisons
3) independent ambiguity subset (temperature-independent)
4) AGR / format-error separation analysis
5) human-ambiguous case study (20-50 samples) with lightweight linguistic feature summary

In [1]:
from __future__ import annotations
from pathlib import Path
import json
import re
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

BASE_DIR = Path('..')
OUT_DIR = BASE_DIR / 'output'
FIG_DIR = OUT_DIR / 'supplementary_figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

VALID_VERDICTS = {'A', 'B', 'C'}
GOLD_MAP = {'model_a': 'A', 'model_b': 'B', 'tie': 'C'}

KEY_TEMPS = [0.01, 1.0, 1.5, 2.0, 3.0]

PIPELINES = {
    'SP-Low (T=0.01,N=1)':    {'temps': [0.01], 'repeats': [0],             'agg': 'single'},
    'SP-Mid (T=1.0,N=1)':     {'temps': [1.0],  'repeats': [0],             'agg': 'single'},
    'Ens-Low (T=0.01,N=10)':  {'temps': [0.01], 'repeats': list(range(10)), 'agg': 'majority_vote'},
    'Ens-High (T=1.5,N=10)':  {'temps': [1.5],  'repeats': list(range(10)), 'agg': 'majority_vote'},
    'Ens-High (T=2.0,N=10)':  {'temps': [2.0],  'repeats': list(range(10)), 'agg': 'majority_vote'},
    'Ens-High (T=3.0,N=10)':  {'temps': [3.0],  'repeats': list(range(10)), 'agg': 'majority_vote'},
    'Ens-Mixed (Mix-T,N=15)': {'temps': [0.5, 1.0, 1.5, 2.0, 3.0], 'repeats': [0, 1, 2], 'agg': 'majority_vote'},
}

print('Imports OK')

Imports OK


In [2]:
def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


def load_runs(file_map: dict[float, list[Path]]) -> pd.DataFrame:
    all_parts = []
    for top_p, files in file_map.items():
        for p in files:
            if not p.exists():
                print(f'[WARN] Missing file: {p}')
                continue
            df = read_jsonl(p)
            df['top_p'] = float(top_p)
            df['source_file'] = p.name
            all_parts.append(df)
    if not all_parts:
        raise FileNotFoundError('No result files found from file_map.')
    out = pd.concat(all_parts, ignore_index=True)
    out['temperature'] = out['temperature'].astype(float)
    out['repeat_id'] = out['repeat_id'].astype(int)
    out['judge_type'] = out['judge_type'].astype(str)
    out['prompt_variant'] = out['prompt_variant'].astype(str)
    out['pairwise_winner'] = out['pairwise_winner'].astype(str)
    out['gold_abc'] = out['gold_winner'].map(GOLD_MAP)
    out['is_parseable'] = (~out['format_error'].astype(bool)) & (out['pairwise_winner'].isin(VALID_VERDICTS))
    out['verdict'] = np.where(out['is_parseable'], out['pairwise_winner'], np.nan)
    return out


RESULT_FILE_MAP = {
    0.95: [
        OUT_DIR / 'results_with_metrics' / 'evaluation_with_metrics_google__gemma-3-27b-it.jsonl',
        OUT_DIR / 'results_with_metrics' / 'evaluation_with_metrics_meta-llama__Llama-3.1-8B-Instruct.jsonl',
    ],
    1.0: [
        OUT_DIR / 'top_p_ablation' / 'evaluation_with_metrics_top_p_1.0_google__gemma-3-27b-it.jsonl',
        OUT_DIR / 'top_p_ablation' / 'evaluation_with_metrics_top_p_1.0_meta-llama__Llama-3.1-8B-Instruct.jsonl',
    ],
}

df_all = load_runs(RESULT_FILE_MAP)
print(f'Loaded rows: {len(df_all):,}')
print('top_p values:', sorted(df_all['top_p'].dropna().unique().tolist()))
print('judge_type:', sorted(df_all['judge_type'].unique().tolist()))
print('prompt_variant:', sorted(df_all['prompt_variant'].unique().tolist()))

[WARN] Missing file: ../output/top_p_ablation/evaluation_with_metrics_top_p_1.0_google__gemma-3-27b-it.jsonl
[WARN] Missing file: ../output/top_p_ablation/evaluation_with_metrics_top_p_1.0_meta-llama__Llama-3.1-8B-Instruct.jsonl
Loaded rows: 360,000
top_p values: [0.95]
judge_type: ['pairwise', 'reference_guided', 'single_answer']
prompt_variant: ['baseline', 'cot']


## 2.1 top_p = 1.0 Ablation (Temperature x top_p)

Main table is built on pairwise + baseline only.

If `top_p=1.0` files are missing, the table still runs for available top_p values and prints warnings.

In [3]:
def verdict_entropy(series: pd.Series) -> float:
    s = series.dropna()
    s = s[s.isin(VALID_VERDICTS)]
    if len(s) == 0:
        return np.nan
    p = s.value_counts(normalize=True)
    return float(-(p * np.log2(p + 1e-12)).sum())


def majority_verdict(series: pd.Series):
    s = series.dropna()
    s = s[s.isin(VALID_VERDICTS)]
    if len(s) == 0:
        return np.nan
    return s.value_counts().idxmax()


pw_base = df_all[(df_all['judge_type'] == 'pairwise') & (df_all['prompt_variant'] == 'baseline')].copy()
grp_cols = ['top_p', 'question_id', 'judge_model', 'temperature']

agg_q = (
    pw_base.groupby(grp_cols)
    .agg(
        final_verdict=('verdict', majority_verdict),
        vote_entropy=('verdict', verdict_entropy),
        format_error_rate=('format_error', lambda x: float(pd.Series(x).astype(float).mean())),
        gold_abc=('gold_abc', 'first'),
    )
    .reset_index()
)
agg_q['AGR'] = ((agg_q['final_verdict'] == agg_q['gold_abc']) & agg_q['final_verdict'].notna()).astype(float)

top_p_main = (
    agg_q[agg_q['temperature'].isin(KEY_TEMPS)]
    .groupby(['temperature', 'top_p'])
    .agg(
        AGR=('AGR', 'mean'),
        Vote_Entropy=('vote_entropy', 'mean'),
        Format_Error_Rate=('format_error_rate', 'mean'),
        N=('AGR', 'count'),
    )
    .reset_index()
    .sort_values(['temperature', 'top_p'])
)

print('=== Temperature x top_p Main Table (Pairwise, Baseline) ===')
display(top_p_main.round(4))

=== Temperature x top_p Main Table (Pairwise, Baseline) ===


,temperature,top_p,AGR,Vote_Entropy,Format_Error_Rate,N
0,0.0100,0.9500,0.5100,0.0129,0.0015,1000
1,1.0000,0.9500,0.4990,0.2388,0.0113,1000
2,1.5000,0.9500,0.4560,0.6061,0.0305,1000
3,2.0000,0.9500,0.4290,0.1031,0.0821,1000
4,3.0000,0.9500,0.4290,0.0757,0.1212,1000


## 2.3 Independent Ambiguity Subset (Temperature-Independent)

This subset uses only external signals independent from judge temperature behavior:
- human tie label (`human_winner == tie`)
- lexical overlap between answer A/B
- response length similarity

In [4]:
WORD_RE = re.compile(r'\w+', re.UNICODE)


def toks(x: str) -> set[str]:
    return set(WORD_RE.findall(str(x).lower()))


def jaccard(a: str, b: str) -> float:
    sa, sb = toks(a), toks(b)
    if not sa and not sb:
        return 0.0
    return len(sa & sb) / max(len(sa | sb), 1)


anchor = (
    pw_base.sort_values(['question_id', 'repeat_id', 'temperature'])
    .drop_duplicates(subset=['question_id'])
    [['question_id', 'answer_a_text', 'answer_b_text', 'human_winner']]
    .copy()
)

anchor['len_a'] = anchor['answer_a_text'].astype(str).str.len()
anchor['len_b'] = anchor['answer_b_text'].astype(str).str.len()
anchor['len_ratio_close'] = (
    (anchor[['len_a', 'len_b']].min(axis=1) / anchor[['len_a', 'len_b']].max(axis=1).clip(lower=1)) >= 0.85
).astype(int)
anchor['lex_overlap'] = anchor.apply(lambda r: jaccard(r['answer_a_text'], r['answer_b_text']), axis=1)
anchor['high_overlap'] = (anchor['lex_overlap'] >= 0.22).astype(int)
anchor['human_tie'] = (anchor['human_winner'].astype(str) == 'tie').astype(int)
anchor['uncertainty_proxy'] = anchor['human_tie'] + anchor['high_overlap'] + anchor['len_ratio_close']

anchor['ambiguous_independent'] = (anchor['uncertainty_proxy'] >= 2)

amb_qids = set(anchor.loc[anchor['ambiguous_independent'], 'question_id'])
nonamb_qids = set(anchor['question_id']) - amb_qids

print(f'Independent ambiguous questions: {len(amb_qids):,} / {anchor.question_id.nunique():,}')
display(anchor[['human_tie', 'high_overlap', 'len_ratio_close', 'uncertainty_proxy', 'ambiguous_independent']].mean().to_frame('mean'))

Independent ambiguous questions: 153 / 500


,mean
human_tie,0.2660
high_overlap,0.5600
len_ratio_close,0.1760
uncertainty_proxy,1.0020
ambiguous_independent,0.3060


## 2.2 Significance / CI for Exp 2-B and 2-C Style Results

In [5]:
def compute_pipeline_metrics(df_sub: pd.DataFrame, agg_mode: str) -> pd.DataFrame:
    rows = []
    for (qid, model), g in df_sub.groupby(['question_id', 'judge_model']):
        valid = g['verdict']
        gold = g['gold_abc'].iloc[0] if g['gold_abc'].notna().any() else np.nan
        if agg_mode == 'single':
            final_v = g['verdict'].iloc[0]
            parseable = pd.notna(final_v)
        else:
            final_v = majority_verdict(valid)
            parseable = pd.notna(final_v)
        agr_all = float(parseable and (final_v == gold)) if pd.notna(gold) else np.nan
        agr_parseable = (1.0 if final_v == gold else 0.0) if (parseable and pd.notna(gold)) else np.nan
        rows.append({
            'question_id': qid,
            'judge_model': model,
            'gold_abc': gold,
            'final_verdict': final_v,
            'parseable_final': float(parseable),
            'AGR_all': agr_all,
            'AGR_parseable_only': agr_parseable,
            'Vote_Entropy': verdict_entropy(valid),
            'CON': (valid == majority_verdict(valid)).mean() if valid.notna().any() else np.nan,
            'ERR_row_mean': float(g['format_error'].astype(float).mean()),
        })
    return pd.DataFrame(rows)


def paired_bootstrap_and_ttest(x: pd.Series, y: pd.Series, n_boot: int = 5000, seed: int = 42) -> dict:
    z = pd.concat([x.rename('x'), y.rename('y')], axis=1).dropna()
    if len(z) == 0:
        return {'n': 0, 'delta': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'p_ttest': np.nan}
    d = (z['x'] - z['y']).to_numpy()
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(d, size=len(d), replace=True).mean() for _ in range(n_boot)])
    t_stat, p_val = stats.ttest_rel(z['x'], z['y'])
    return {
        'n': int(len(d)),
        'delta': float(d.mean()),
        'ci_low': float(np.percentile(boot, 2.5)),
        'ci_high': float(np.percentile(boot, 97.5)),
        'p_ttest': float(p_val),
    }


def build_pipe_results(df_variant: pd.DataFrame, subset_qids: set[int]) -> dict[str, pd.DataFrame]:
    out = {}
    sub0 = df_variant[df_variant['question_id'].isin(subset_qids)]
    for name, cfg in PIPELINES.items():
        x = sub0[sub0['temperature'].isin(cfg['temps']) & sub0['repeat_id'].isin(cfg['repeats'])]
        out[name] = compute_pipeline_metrics(x, cfg['agg'])
    return out


pw_base = pw_base.copy()
pw_cot = df_all[(df_all['judge_type'] == 'pairwise') & (df_all['prompt_variant'] == 'cot') & (df_all['top_p'] == 0.95)].copy()

base_amb = build_pipe_results(pw_base[pw_base['top_p'] == 0.95], amb_qids)
base_non = build_pipe_results(pw_base[pw_base['top_p'] == 0.95], nonamb_qids)
cot_amb = build_pipe_results(pw_cot, amb_qids)
cot_non = build_pipe_results(pw_cot, nonamb_qids)

# Exp 2-B style: pipeline AGR_all vs SP-Low within same subset/prompt (paired)
sig_2b = []
for prompt_name, res_dict in [('baseline', base_amb), ('cot', cot_amb)]:
    ref = res_dict['SP-Low (T=0.01,N=1)'].set_index(['question_id', 'judge_model'])['AGR_all']
    for pname, r in res_dict.items():
        cur = r.set_index(['question_id', 'judge_model'])['AGR_all']
        st = paired_bootstrap_and_ttest(cur, ref)
        st.update({'prompt_variant': prompt_name, 'subset': 'Ambiguous', 'pipeline': pname, 'vs_ref': 'SP-Low'})
        sig_2b.append(st)

sig_2b_df = pd.DataFrame(sig_2b).sort_values(['prompt_variant', 'pipeline']).reset_index(drop=True)
print('=== Significance for Exp 2-B style contrasts (Ambiguous subset) ===')
display(sig_2b_df.round(4))

# Exp 2-C style: CoT - Baseline per pipeline, same subset, paired by question-model
sig_2c = []
for subset_name, base_res, cot_res in [('Ambiguous', base_amb, cot_amb), ('Non-Ambiguous', base_non, cot_non)]:
    for pname in PIPELINES.keys():
        b = base_res[pname].set_index(['question_id', 'judge_model'])['AGR_all']
        c = cot_res[pname].set_index(['question_id', 'judge_model'])['AGR_all']
        st = paired_bootstrap_and_ttest(c, b)
        st.update({'subset': subset_name, 'pipeline': pname, 'contrast': 'CoT - Baseline'})
        sig_2c.append(st)

sig_2c_df = pd.DataFrame(sig_2c).sort_values(['subset', 'pipeline']).reset_index(drop=True)
print('=== Significance for Exp 2-C style contrasts (CoT - Baseline) ===')
display(sig_2c_df.round(4))

=== Significance for Exp 2-B style contrasts (Ambiguous subset) ===


,n,delta,ci_low,ci_high,p_ttest,prompt_variant,subset,pipeline,vs_ref
0,306,0.0752,0.0359,0.1144,0.0003,baseline,Ambiguous,"Ens-High (T=1.5,N=10)",SP-Low
1,306,0.2745,0.2092,0.3399,0.0000,baseline,Ambiguous,"Ens-High (T=2.0,N=10)",SP-Low
2,306,0.2745,0.2092,0.3399,0.0000,baseline,Ambiguous,"Ens-High (T=3.0,N=10)",SP-Low
3,306,0.0000,0.0000,0.0000,NaN,baseline,Ambiguous,"Ens-Low (T=0.01,N=10)",SP-Low
4,306,0.0294,0.0000,0.0588,0.0494,baseline,Ambiguous,"Ens-Mixed (Mix-T,N=15)",SP-Low
5,306,0.0000,0.0000,0.0000,NaN,baseline,Ambiguous,"SP-Low (T=0.01,N=1)",SP-Low
6,306,-0.0131,-0.0261,-0.0033,0.0453,baseline,Ambiguous,"SP-Mid (T=1.0,N=1)",SP-Low
7,306,0.2810,0.2157,0.3464,0.0000,cot,Ambiguous,"Ens-High (T=1.5,N=10)",SP-Low
8,306,0.2712,0.2026,0.3366,0.0000,cot,Ambiguous,"Ens-High (T=2.0,N=10)",SP-Low
9,306,0.2908,0.2222,0.3562,0.0000,cot,Ambiguous,"Ens-High (T=3.0,N=10)",SP-Low


=== Significance for Exp 2-C style contrasts (CoT - Baseline) ===


,n,delta,ci_low,ci_high,p_ttest,subset,pipeline,contrast
0,306,0.1275,0.0621,0.1961,0.0003,Ambiguous,"Ens-High (T=1.5,N=10)",CoT - Baseline
1,306,-0.0817,-0.1176,-0.0490,0.0000,Ambiguous,"Ens-High (T=2.0,N=10)",CoT - Baseline
2,306,-0.0621,-0.0915,-0.0327,0.0001,Ambiguous,"Ens-High (T=3.0,N=10)",CoT - Baseline
3,306,-0.0686,-0.1078,-0.0294,0.0010,Ambiguous,"Ens-Low (T=0.01,N=10)",CoT - Baseline
4,306,0.0850,0.0196,0.1503,0.0113,Ambiguous,"Ens-Mixed (Mix-T,N=15)",CoT - Baseline
5,306,-0.0784,-0.1209,-0.0392,0.0002,Ambiguous,"SP-Low (T=0.01,N=1)",CoT - Baseline
6,306,-0.0392,-0.0882,0.0098,0.1089,Ambiguous,"SP-Mid (T=1.0,N=1)",CoT - Baseline
7,694,-0.1153,-0.1499,-0.0821,0.0000,Non-Ambiguous,"Ens-High (T=1.5,N=10)",CoT - Baseline
8,694,0.0086,-0.0058,0.0231,0.2571,Non-Ambiguous,"Ens-High (T=2.0,N=10)",CoT - Baseline
9,694,0.0072,-0.0072,0.0231,0.3535,Non-Ambiguous,"Ens-High (T=3.0,N=10)",CoT - Baseline


## 2.4 AGR / Format-Error Separation Analysis

Report all-sample AGR, parseable-only AGR, and format error rate together.

In [6]:
def summarize_agr_err(pipe_res: dict[str, pd.DataFrame], subset_name: str, prompt_name: str) -> pd.DataFrame:
    rows = []
    for pname, dfm in pipe_res.items():
        fer = 1.0 - dfm['parseable_final'].mean()
        rows.append({
            'Subset': subset_name,
            'Prompt': prompt_name,
            'Pipeline': pname,
            'AGR_all_sample': dfm['AGR_all'].mean(),
            'AGR_parseable_only': dfm['AGR_parseable_only'].mean(),
            'Format_Error_Rate_final': fer,
            'N': len(dfm),
        })
    return pd.DataFrame(rows)

sep_tbl = pd.concat([
    summarize_agr_err(base_amb, 'Ambiguous', 'baseline'),
    summarize_agr_err(cot_amb, 'Ambiguous', 'cot'),
    summarize_agr_err(base_non, 'Non-Ambiguous', 'baseline'),
    summarize_agr_err(cot_non, 'Non-Ambiguous', 'cot'),
], ignore_index=True)

print('=== AGR vs Parseability Separation Table ===')
display(sep_tbl.round(4))

=== AGR vs Parseability Separation Table ===


,Subset,Prompt,Pipeline,AGR_all_sample,AGR_parseable_only,Format_Error_Rate_final,N
0,Ambiguous,baseline,"SP-Low (T=0.01,N=1)",0.2843,0.2843,0.0000,306
1,Ambiguous,baseline,"SP-Mid (T=1.0,N=1)",0.2712,0.2730,0.0065,306
2,Ambiguous,baseline,"Ens-Low (T=0.01,N=10)",0.2843,0.2843,0.0000,306
3,Ambiguous,baseline,"Ens-High (T=1.5,N=10)",0.3595,0.3595,0.0000,306
4,Ambiguous,baseline,"Ens-High (T=2.0,N=10)",0.5588,0.5588,0.0000,306
5,Ambiguous,baseline,"Ens-High (T=3.0,N=10)",0.5588,0.5588,0.0000,306
6,Ambiguous,baseline,"Ens-Mixed (Mix-T,N=15)",0.3137,0.3137,0.0000,306
7,Ambiguous,cot,"SP-Low (T=0.01,N=1)",0.2059,0.2283,0.0980,306
8,Ambiguous,cot,"SP-Mid (T=1.0,N=1)",0.2320,0.2536,0.0850,306
9,Ambiguous,cot,"Ens-Low (T=0.01,N=10)",0.2157,0.2332,0.0752,306


## 2.5 Human-Ambiguity Case Study (20-50 Samples)

Uses human tie questions as independent human-ambiguous set; prioritizes cases where high-T ensemble helps over SP-Low.

In [7]:
def clip_text(x: str, n: int = 240) -> str:
    x = str(x).replace('\n', ' ').strip()
    return x if len(x) <= n else x[:n] + ' ...'


human_amb_qids = set(anchor.loc[anchor['human_tie'] == 1, 'question_id'])

sp_low = base_amb['SP-Low (T=0.01,N=1)'].copy()
ens_high = base_amb['Ens-High (T=2.0,N=10)'].copy()

m = (
    sp_low.set_index(['question_id', 'judge_model'])[['AGR_all', 'final_verdict', 'Vote_Entropy']]
    .rename(columns={'AGR_all': 'AGR_sp_low', 'final_verdict': 'v_sp_low', 'Vote_Entropy': 'ent_sp_low'})
    .join(
        ens_high.set_index(['question_id', 'judge_model'])[['AGR_all', 'final_verdict', 'Vote_Entropy']]
        .rename(columns={'AGR_all': 'AGR_ens_high', 'final_verdict': 'v_ens_high', 'Vote_Entropy': 'ent_ens_high'})
    )
    .reset_index()
)
m['improved'] = (m['AGR_ens_high'] > m['AGR_sp_low'])
m = m[m['question_id'].isin(human_amb_qids)].copy()

ans_ref = (
    pw_base[pw_base['question_id'].isin(human_amb_qids)]
    .sort_values(['question_id', 'repeat_id', 'temperature'])
    .drop_duplicates(subset=['question_id'])
    [['question_id', 'model_a', 'model_b', 'answer_a_text', 'answer_b_text', 'reference_answer', 'gold_winner']]
)

case_pool = m.merge(ans_ref, on='question_id', how='left')
case_pool = case_pool.sort_values(['improved', 'ent_ens_high'], ascending=[False, False])

N_CASES = 30
case_show = case_pool.head(N_CASES).copy()
for c in ['answer_a_text', 'answer_b_text', 'reference_answer']:
    case_show[c + '_snippet'] = case_show[c].apply(lambda x: clip_text(x, n=240))

case_cols = [
    'question_id', 'judge_model', 'gold_winner', 'v_sp_low', 'v_ens_high',
    'AGR_sp_low', 'AGR_ens_high', 'improved', 'ent_sp_low', 'ent_ens_high',
    'model_a', 'model_b',
    'answer_a_text_snippet', 'answer_b_text_snippet', 'reference_answer_snippet',
]

print(f'Human-ambiguous pool size: {len(case_pool):,}')
print(f'Showing top {len(case_show)} cases for qualitative review.')
display(case_show[case_cols])

save_path = OUT_DIR / 'supplementary_figures' / 'case_study_human_ambiguous_top30.csv'
case_show[case_cols].to_csv(save_path, index=False)
print(f'Saved: {save_path}')

Human-ambiguous pool size: 216
Showing top 30 cases for qualitative review.


,question_id,judge_model,gold_winner,v_sp_low,v_ens_high,AGR_sp_low,AGR_ens_high,improved,ent_sp_low,ent_ens_high,model_a,model_b,answer_a_text_snippet,answer_b_text_snippet,reference_answer_snippet
143,371,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.7642,google/gemma-3-12b-it,meta-llama/Llama-3.1-8B-Instruct,**Final Choice:** A. ['Fermions have antisymme...,A. ['Fermions have antisymmetric wave function...,Final Answer: A Analysis: By the spin–statist...
151,418,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.7642,google/gemma-3-12b-it,Qwen/Qwen2.5-14B-Instruct,**Final Choice: A** **Reason:** Market social...,The correct answer is: **Option A: An economi...,Final Answer: A Analysis: Market socialism is...
161,429,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.7642,Qwen/Qwen2.5-14B-Instruct,meta-llama/Llama-3.1-8B-Instruct,To find the work done by the force \(\mathbf{F...,"To find the work done, we need to calculate th...",Final Answer: 144 J Analysis: Displacement = ...
169,437,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.7642,meta-llama/Llama-3.1-8B-Instruct,google/gemma-3-12b-it,My final choice is: D. ['The female bacterium...,**Final Choice: D. 'The female bacterium becom...,Final Answer: D. The female bacterium becomes ...
189,458,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.7642,google/gemma-3-12b-it,meta-llama/Llama-3.1-8B-Instruct,**Final Choice:** B. 'These elements are abund...,Option: B. These elements are abundant in livi...,Final Answer: B. These elements are abundant i...
55,120,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.5436,gpt-4,gpt-3.5-turbo,If it's not a binary tree but a general tree w...,"If it is not a binary tree, the approach to fi...",Short answer: the algorithm generalizes direct...
103,267,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.5436,gpt-3.5-turbo,vicuna-13b-v1.2,"If it is not a binary tree, the approach to fi...","If the tree is not a binary tree, then the pro...",First clarify terminology: in most literature ...
141,370,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.5436,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-14B-Instruct,To find the amount of lead absorbed in the lun...,"To solve this problem, let's break it down ste...",Final Answer: H. 15.9 μg Analysis: Inhaled le...
159,425,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.5436,Qwen/Qwen2.5-14B-Instruct,meta-llama/Llama-3.1-8B-Instruct,The farmer's best defense in this scenario wou...,"Option: ""The farmer's promise to survey the tr...",Final Answer: H Analysis: Specific performanc...
167,434,meta-llama/Llama-3.1-8B-Instruct,tie,A,C,0.0000,1.0000,True,-0.0000,0.5436,meta-llama/Llama-3.1-8B-Instruct,Qwen/Qwen2.5-14B-Instruct,"Option: 'D. In a race-notice jurisdiction, the...",The correct answer is: **Option D: In a race-...,Final Answer: A Analysis: The statute is a no...


Saved: ../output/supplementary_figures/case_study_human_ambiguous_top30.csv


In [8]:
# Lightweight linguistic feature summary: improved vs not improved
HEDGE_RE = re.compile(r'\b(maybe|perhaps|possibly|might|could|likely|uncertain)\b', re.IGNORECASE)
NUM_RE = re.compile(r'\d')
CODE_RE = re.compile(r'```|def\s+|class\s+|import\s+', re.IGNORECASE)

def ling_features(a: str, b: str) -> dict:
    ta = str(a)
    tb = str(b)
    both = ta + ' ' + tb
    return {
        'avg_len': (len(ta) + len(tb)) / 2.0,
        'len_ratio_close': (min(len(ta), len(tb)) / max(len(ta), len(tb), 1)) >= 0.85,
        'lex_overlap': jaccard(ta, tb),
        'has_hedge': bool(HEDGE_RE.search(both)),
        'has_number': bool(NUM_RE.search(both)),
        'has_code_like': bool(CODE_RE.search(both)),
    }

feat_rows = []
for _, r in case_pool.iterrows():
    d = ling_features(r.get('answer_a_text', ''), r.get('answer_b_text', ''))
    d['improved'] = bool(r['improved'])
    feat_rows.append(d)

feat_df = pd.DataFrame(feat_rows)

summary_ling = feat_df.groupby('improved').agg({
    'avg_len': 'mean',
    'len_ratio_close': 'mean',
    'lex_overlap': 'mean',
    'has_hedge': 'mean',
    'has_number': 'mean',
    'has_code_like': 'mean',
}).rename(index={True: 'High-T improved', False: 'Not improved'})

print('=== Linguistic feature profile on human-ambiguous pool ===')
display(summary_ling.round(4))

=== Linguistic feature profile on human-ambiguous pool ===


,avg_len,len_ratio_close,lex_overlap,has_hedge,has_number,has_code_like
improved,,,,,,
Not improved,769.1193,0.2844,0.3637,0.1743,0.6055,0.0917
High-T improved,776.7383,0.2710,0.3525,0.1776,0.5981,0.0935


## Optional: Minimal top_p=1.0 Run Template

Use this only when `top_p=1.0` files do not exist yet. It runs a small key-temperature setup.

In [ ]:
# Fill in your endpoint/model, then run in terminal (recommended):
# uv run llmjudge run -m <MODEL_NAME> -u <BASE_URL> -b vllm -s <SIZE_LABEL> \
#   -t 0.01,1.0,1.5,2.0,3.0 -j pairwise -p baseline,cot -r 10 -n 300 --top-p 1.0 \
#   -o output/top_p_ablation/<RUN_NAME>

print('Template only. After run finishes, export to JSONL with metrics and add files to RESULT_FILE_MAP[1.0].')